# Welcome to DRAGON Tutorials 

This is the first of a series of tutorials. Each is designed as a sequence of connected steps that build on one another, gradually leading you toward a complete solution, e.g., a simple two-window drift detector in this tutorial. Each step is accompanied by a short "mini tutorial" that introduces relevant concepts in a broader context. These explanations often go beyond what is strictly required for the task at hand, so part of the exercise is identifying and transferring the ideas that are most useful for the current step.

After each step, you'll find a detailed "unit test." These tests not only check whether your solution works but also provide guidance by making expectations explicit, helping you reflect on and refine your approach. The exact workings of those tests depend on the exercise; usually, they provide insight into what your implementation does and what it should do.

If you get stuck, a sample solution is available for every step in a separate file. This allows you to get back on track. You’re encouraged to give each task a solid attempt first—even if it feels challenging—but you can rely on the solutions as a safety net when needed.


In [1]:

# These are just some imports, run them once so everything works properly
from dragon import parse
from dragon.detectors import MMD
from dragon.stream import GrowingWindow
from dragon.stream import SlidingWindow
from dragon.stream import Pipeline
from dragon.stream import WindowChunker
from dragon.util.Stream import Stream, from_file
from tutorial.util import check1_1, check1_2, check1_3, check1_4, check1_5, check1_6, window_filling, pipeline_animation, \
    stream_generator_animation

# Tutorial 1 -- Basics: Data, Streams, Flows

In this first tutorial, we will build a simple drift detector using DRAGON modules/tools and run it on a toy-example data stream. We proceed in three steps:
1. Loading the data
2. Preparing the memory management: windows and data flow
3. Set up the actual detector
4. Execute a test run

After that, the most basic version is finished. We then extend on this and explore a bit further 

5. Set up a more complex memory management system: additional options of windowing and data flow are discussed
6. Use DRAGON's Window Annotation Language: setting up more complex systems more easily

## Step 1 — Getting the Data Ready

**You are learning:** You are learning how DRAGON represents streams and how to create them from common sources.

**Your task:** Use the DRAGON tools to create a DRAGON Stream-object by loading the file `test_data.csv`

<details>
<summary>Open Tutorial</summary>
<div class="toggle-box" id="box1">
  <div class="content" style="padding-left: 3em;">
    <h3>Mini Tutorial on DRAGON stream generation:</h3>
    DRAGON offers a variety of options to work with data. Internally, data points are represented by Python <code>String -> Object</code> dictionaries. Collections of data -- what amounts to datasets in the batch setting or stream benchmarks in usual stream learning -- are represented by <code>Stream</code>-objects.
    <h4>Creating a <code>Stream</code>: Options and Pathways</h4>
    There are a few ways to create a <code>Stream</code>. In DRAGON, two main pathways exist:
    <ol>
      <li> By wrappering a container or generator like a Pandas <code>DataFrame</code>, an Numpy array, a river <code>Dataset</code>, or a simple Python <code>list</code></li>
        <li>By using a <code>ConceptBank</code> for more advanced and controlled setups</li>
      </ol>
      Here, we will only focus on the first case and leave the second for more advanced tutorials.
      <ul>
          <li><b>Pandas <code>DataFrame</code></b> can be used directly in the constructor of a <code>Stream</code>. In fact, it is probably best to think of <code>Stream</code> as DRAGON wrappers of <code>DataFrames</code></li>
          <li><b>Numpy arrays</b> are probably the first or second most common way to store data for ML. The <code>from_numpy_array</code> method is an easy way to wrap arrays. Attention: while not required, it is advised to provide column names -- feature 1 to 5 is not very informative as a name</li>
          <li><b>River streams</b> can be turned into <code>Stream</code> by using the <code>from_river</code> method. Attention: in case you use a synthetic datastream, i.e., SEA, STAGGER, Agrawal, etc., you need to specify a stream length; for most other river streams, this is not necessary</li>
          <li><b>Files</b> can be conveniently loaded using the <code>from_file</code> function. Attention: The function only wraps Pandas file loading with the loader being selected on a file name extension basis, so make sure the file has the correct name extension (an additional <code>.zip</code>/<code>.gzip</code>/<code>.xz</code>/... for compressed files is always fine)</li>
          <li><b>Lists and generators</b> (or more generally iterateables) can also be used via the <code>from_python</code> method. As for river streams, if <code>len</code> is not supported, a length must be specified. The expected object type are dicts <code>String -> Object</code></li>
      </ul>
    <h4>An Example</h4>
  </div>
</div>

<pre><code class="language-python">
n_length, n_dimensions = 500, 2

# Creating the data as a numpy array
X = np.random.normal(size=(n_length,n_dimensions))
X[n_length//2:] += 1 #adding some drift in the middle of the stream

# Creating the StreamGenerator
stream_a = from_numpy_array(X,"feature 1","feature 2")

# Using River
river_stream = SEA()
stream_b = from_river(river_stream, n_length=n_length)

my_stream_data = stream_a.take(50)

</code></pre>

</details>



Your code for loading `test_data.csv` goes here:

Self check your code: (Human-based Unit-Test)

In [2]:
# add your code here

my_stream = from_file("test_data.csv")  # your object goes here

# Use this test, when you are ready
check1_1(my_stream)


This test checks, if you did load the data correctly.
Well done! You did successfully load the data correctly! On to the next step!


Now that we have the data, let us build the drift detector to process it. In many cases, a drift detector consists of 
 1) A memory management system is usually in the form of sliding windows and
 2) An actual testing procedure

We proceed with setting up the windows; the test will follow in step 3. 

## Step 2 — Memory Management: Windows & Data Flow

**You are learning:**

Setting up the memory management system consists of two steps: 
 1. Creating the window objects
 2. Linking the windows to direct data flow through them

**Your task:** Use the DRAGON tools to create two consecutive sliding windows of 250 samples in length. We refer to the first as "current" and the second as "reference" window
<details>
<summary>Open Tutorial</summary>


<div class="toggle-box" id="box1">
  <div class="content" style="padding-left: 3em;">
    <h3>Mini Tutorial on DRAGON streaming objects, pipelines, and memory management:</h3>
    In stream learning and processing, not only is the data streaming, but also the processing itself. Instead of fixed data batches, dynamically updated (sliding) windows are one of the go-to strategies to store data samples for further processing. In this mini tutorial, we will discuss the basics of how to create such dynamic processing and storing pipelines. 
    <h4>Data Flow in DRAGON</h4>
    In DRAGON, data flow is realized via events that "carry" the data points from processing step to processing step. The steps themselves are realized via <code> StreamObject </code>s. There are four main kinds of <code>StreamObject</code>:
    <ol>
        <li><code>Map</code>s process the data as it arrives using a specified function; data is not stored</li>
        <li><code>Filter</code> is used to remove points from the processing; only data that meets a specified condition is ejected, otherwise it is removed. The data points themselves are left untouched</li>
        <li><code>Chunker</code> collects data points until a condition (usual batch size) is met and then emits the entire batch at once. This can be used to optimize the control flow</li>
        <li><b>Windows</b> are a group of <code>StreamObjects</code>. Windows store parts of the streams to be used for further processing; they eject data once it is no longer stored, usually because too much new data was received or because the window was reset. The pre-implemented windows are 
            <ul><li><code> SlidingWindow </code> which has a fixed length and slides along the stream, data points are ejected, in order of receiving them</li>
                <li><code>StaticWindow</code> contain a fixed amount of samples. They are filled with data once and no longer change. Data that is received once the window is filled is directly ejected</li>
                <li>GrowingWindow</code> keep a representative sample of the data observed thus far (see <a href="https://en.wikipedia.org/wiki/Reservoir_sampling">Reservoir Sampling</a>). Data is ejected in an randomized order</li></ul></li>
    </ol>
    <h4>Linking <code>StreamObject</code>s</h4>
    To link several of those predefined building blocks into a stream processing pipeline, DRAGON offers two options
    <ol>
        <li><code>Pipelines</code>s arrange the objects one after the other: Data fed into the pipeline is first fed into the first component; data ejected from the first component is fed into the second, and so on; finally, data ejected from the last component is ejected from the pipeline.</li>
        <li><code> Forks </code>s are used to perform parallel processing of data; data fed into a fork is fed into each of its components in parallel. <b>Attention</b> once split, the pathways cannot be merged again!</li>
    </ol>
    <h4>Accessing a Window's Data</h4>
    To access the data stored in a window, the <code>get_data</code> method can be used. This method is also implemented by <code>Pipeline</code>s; in this case it returns the data of all windows it contains. 
    <br>
    For easier handling, DRAGON supports naming, which applies to all windows and <code>Pipeline</code>s; those objects can be named by setting the name parameter in the constructor or by calling the <code>set_name</code> method. A named object inside a pipeline can be accessed via index notation, i.e., <code>my_pipe["my_favorite_window"]</code>. Note that pipes can be used within other pipes. 
    Aside from manual naming, pipes and forks also ensure automatic naming, e.g., the first accessible component in a pipeline can always be accessed by the name <code>W_1</code>.
    <h4>An Example</h4>

    <pre><code class="language-python">
        # multiply any incoming data by 10 and take a look at the data
        my_pipe1 = Pipeline(Peek(), Map(lambda x: {'value': x.get('value', 0) * 10}))

        # a pipeline containing a sliding window
        my_pipe2 = Pipeline(SlidingWindow(max_size=100))

    </code></pre>

   <!--p>Topics:
        <ul>
            <li> Windows contain some of the datapoints that can be queried </li>
            <li>windows can be named</li>
            <li>Window pipelines group sequences of windows</li>
            <li>Example: [cur: slide] -> [ref: static]; access data via api</li>
        </ul>
    </p-->

  </div>
</div>
</details>
Your code goes here:

In [3]:
my_pipe = Pipeline(SlidingWindow(max_size=250), SlidingWindow(max_size=250))  # your pipe goes here


Self check your code: (Human-based Unit-Test)

In [4]:
# Use this test, when you are ready
check1_2(my_pipe)
# If you need a deeper insight of the windows filling, take a look at this animation
# window_filling()
# maybe a look at the pipeline data flow helps too!
# pipeline_animation()

This test checks, if your pipeline is correctly implemented.
Great! Your pipeline is correct!


## Step 3 — Setting up the Test

**You are learning:** How to build a simple drift detector using DRAGON 

A very simple testing procedure is to apply a two-sample test -- like the Maximum Mean Descripancy (MMD) kernel two-sample test [learn more](https://www.frontiersin.org/journals/artificial-intelligence/articles/10.3389/frai.2024.1330257/pdf) -- to compare the distribution in the current and reference window. If there is drift the test should be able to find a difference in distributions. Every time a new data point is added, the test is executed; should the test find suffcient evidence for a drift raises an alert.  

**Your task:** Use the DRAGON tools to create a two-window-based drift detector using MMD as testing procedure. On detection alert drift (via print). 
<details>
<summary>Open Tutorial</summary>
<div class="toggle-box" id="box1">
  <div class="content" style="padding-left: 3em;">
    <h3>Mini Tutorial on building drift detectors with DRAGON:</h3>
    DRAGON implements various drift detectors that are directly based on statistical tests. Here, the actual testing for drift happens in an offline batch fashion which is applied to windows to make the test a stream-ready drift detector. There are two main categories of drift detectors: those that compare the distributions of two (explicitely or implicitely) stored samples -- usually either data or model losses -- implicitly assuming no drift within the windows themselfs, and those that actively search for drifts within a single sample. Here, we will focus on the first chategory and will refer to them as 2-window-based detectors.
<br>
To run the inner test of the detector requires two samples, those need to be provided at creation time in form of windows (or more generally, objects that implement the <code>DataProvider</code> interface). Here, the afformentioned nameing of windows can come in handy. Most detectors also take additional arguments like decision thresholds, kernels, models, etc.
<br>
The detector itself is passive, the perform the actual detection the <code>detect</code> method must be called. This will execute the test on the data currently contained in the windows and call an <code>drift_detectored</code>-event in case of detection which contains further informaiton on the detection result. The most common and natural way to implement a drift detector is to make it listen to its input windows; in this case the detection will be exectued every time new data is received.
<br>
To take appropriate action in case of detection one listens to the detector which raises events in case drift was detected. Using this one can alert the user by printing on the consol, trigger explanations, or also control the dataflow in the windows, e.g., by resetting the reference window, or similar.
<h4>An Example</h4>
The following is an example of a two-window drift detector based on the feature-wise Kolmogorov-Smirnov test. It uses two consecutive sliding windows and chekcs for drift every time the reference window changes
<pre><code class="language-python">
p = Pipeline(SlidingWindow(100, name="ref"),SlidingWindow(100, name="cur"))
ks = MultiDimKolmogorofSmirnov(p["ref"], p["cur"])

p["ref"].add_state_listener(ks.detect) ## Test every time the reference window's content chagnes (may lead to instable detections at the beginning when the reference window still only contains few data points)
</pre></code>
Another option that is better suited if we expect slow drift is to use a static reference window which is filled once and then remaines the same. The easiest way to implement this is to simply change the class from sliding to static window. However, as the reference window will no longer change its state once it is filled we have to either listen to the current windows state or the reference windows outflow. Here, listening to the outflow has the advantage that we wait till both windows are filled and the test thus works stable.
<pre><code class="language-python">
p = Pipeline(StaticWindow(100, name="ref"),SlidingWindow(100, name="cur"))
ks = MultiDimKolmogorofSmirnov(p["ref"], p["cur"])

#OPTION listen to current windows state: p["cur"].add_listener(lambda evt: ks.detect() if len(p["ref"]) > 50 else None) ## Tests once the sliding window is filled
p["ref"].add_outflow_listener(ks.detekt)
</pre></code>
However, once drift is detected, since the content of the static window will not change this will lead to repeated detections. To avoid this we have to fill the static window with samples of the current distribution. This can be done by defining the reset of the pipe and then call reset on detected events.
<pre><code>
p.set_resetcode("ref,cur") #first empty the reference window, then empty the current window, this will automatically push the samples form the current into the reference window
ks.add_listener(p.reset)
</code></pre>

As an advanced example we consider equal size consecutive sliding windows. Such have the advantage that for most two-sample tests the certainty depends on the smaller sample. By using the same size for both windows, letting them fill up at the same rate, we thus reach higher detection power with fewer samples. This can be implemented as follows:
<pre><code>
p["cur"].set_size(1)
p["cur"].add_outflow_listener(lambda evt: p["cur"].set_size(min(p["ref"].max_size, len(p["ref"])+1)))
</code></pre>
ATTENTION: this only adapts the window size at the very beginning. It is not meant to be combined with resets. If you should wish to combine consecutive sliding windows with resets, it is better to only reset the current window; a reset of the reference window and thus a equal size filling is not neccessary.

</details>
<!--p>Topics:
        <ul>
            <li>Short description of 2-window-based drif detectors</li>
            <li>Example: KS test</li>
        </ul></p-->

  </div>
</div>

Your code goes here:

In [5]:
p = Pipeline(SlidingWindow(max_size=250, name="ref"), SlidingWindow(max_size=250, name="cur"))
my_detector = MMD(p["ref"], p["cur"])

Self check your code: (Human-based Unit-Test)

In [6]:
# Use this check, when you are ready!
check1_3(my_detector)

This test checks, if this detector is correctly implemented.
Great! Your detector is ready!


## Step 4 — Running the Example

**You are learning:** How to apply the stream processing pipe to a stream. By the end of this Step you will have your first working example! :)

**Your task:** Let the stream you created in step 1 run though your processing pipeline. 
<details>
<summary>Open Tutorial</summary>
<div class="toggle-box" id="box1">
  <div class="content" style="padding-left: 3em;">
    <h3>Mini Tutorial on running your DRAGON pipeline:</h3>
    As everything else, also streams are event based. Instead of a container or generator, better think of a DRAGON's <code>StreamGenerator</code>-object as a device emulator that simulates a device receiving the data stored inside it. Therefore the workflow is slightly different from other packages like river. While in river you take samples from the stream and then push them in the processing pipeline, in DRAGON you let the pipeline listen to the stream which fires <code>new_data_point</code>-events. In case of a <code>StreamGenerator</code> you then need to call <code>gen_stream()</code> to "run the simulation."
    The StreamGenerator generally fires its events regardless of any listener. So to receive events make sure your <code>StreamObject</code> listens to the stream using the <code>add_listener</code> method.

   As you can see, many gears mesh together. To get a better idea of how the events work in general lets take a look at this picture.

<img src="img/event_flow.png">


  </div>
</div>
<h4>An Example</h4>
<pre><code class="language-python">
my_stream = StreamGenerator(my_data_set)
my_pipe = Pipeline(SlidingWindow(max_size=250), SlidingWindow(max_size=250))
my_stream.add_listener(my_pipe)
my_stream.gen_stream()
</code></pre>
<div class="toggle-box" id="box1">
  <div class="content" style="padding-left: 3em;">


  </div>
</div>
</details>
Your code goes here:

In [7]:
my_stream.add_listener(my_pipe)
my_stream.run()


Self check your code: (Human-based Unit-Test)

In [8]:
# Use this check, when you are ready!
check1_4(my_pipe)

This test checks if the stream actually flowed through your pipeline.

Success! Your pipeline processed the stream and currently holds  500  samples in memory.


***Congratulations!* You implemented your first drift detector in DRAGON :-)**

In the following, we will look at a few more advanced window setups.

## Step 5 — Advanced Windows and Control Flow

**You are learning** about other kinds of windows implemented in DRAGON and more advanced flow and data management.

In some setups, simple sliding windows are not enough to take care of certain situations. Rather we need a more complex window and data fow control to archive optimal results. In the following we create such a setup.

**Your task:** Replace your current sequence of windows as described by the following pipeline

<img src="img/step5_pipe.png">

For this implement a pipline that processes the data as follows:
1. First split the data into chunks of 10 samples -- this ensures that you have control on how often the test is executed AND you make sure that you don't feed too large chunks into the detector
2. Use a 250 samples long sliding "current" window -- this is just the standard setup 😉
3. The "reference" window consists of two consecutive windows: The first is a 100 samples long sliding window followed by a 150 samples growing window -- the sliding allows fast reaction, the growing keeps track of the longer past
4. If the detector alerts the reference window is flushed clean, then the data of the current window is pushed into it

<details>
<summary>Open Tutorial</summary>
<div class="toggle-box" id="box1">
  <div class="content"  style="padding-left: 3em;">
    <h3>Mini Tutorial Advanced Data Flow Control in Pipes:</h3>

In step 2 we already discussed various way types of windows and their properties. We will not have a closer look at how to model the data flow during rests of pipes. This is paramount to ensure valid system behaviour in more compelx setups.
<h4>Resetting a Pipe</h4>
Pipes usually reset by running reset of its components starting form the first going to the last resulting in a complete drainage of the pipe. This behavior is not necessarily desirable. For this reason, pipes offer an easy way to "overload" this standard procedure. This involves mainly 3 commands that are executed on the sub-window in user defined order
<ul>
<li><code>reset</code> resets the window according to its specification (usually flushing out the complete content via the standard <code>out_flow</code> events)
<li><code>close</code> blocks the window from processing data; if the window is closed all data that is passt to it via is directly pushed out via <code>out_flow</code> events independent of the current window state
<li><code>open</code> sets the window back from the closed state to the usual mode of operation
</ul>
To simplify usage DRAGON implements a simple interface language that works via the naming convention which is specified as follows
<pre>
S ::= &lt;name&gt;["," S] | &lt;name&gt; "(" S ")" [S]
</pre>
Here
<ul>
<li> <code>&lt;name&gt;</code> references the windows relative to the specifying pipe, i.e., if we set <code>p.set_rest_script("W_1,W_2")</code> this is interpreted as <code>p["W_1"].reset(); p["W_2"].reset()</code>. Names are expected to be accepted by the regular expression [A-Za-z0-9_]+ or specified with quotation marks <code>"..."</code> which can be escaped via <code>\"</code> as usual.
<li> <code>&lt;name&gt;(...)</code> is interpreted as a close-open-pair, i.e., <code>p.set_rest_script("W_1(W_2)")</code> is interpreted as <code>p["W_1"].close(); p["W_2"].reset(); p["W_1"].open()</code>
<li><code>,</code> serves as a separator between names -- if no close-open-instruction is given every name is interpreted as calling <code>reset</code>
</ul>
<h4>An Example</h4>
Consider for example the setup for a two-window-based drift detector:
<pre>
[reference: fixed 250] <- [current: slide 250] <-o
</pre>
which may be implemented as
<code>
p = Pipe(FixedWindow(250,name="reference"),Slidingwindow(250,name="current"))
dd = MMD(p["current"],p["reference"])
</code>
In case of a drift we need to update the reference window so that it contains samples of the current and not the outdated distribution. To achieve this we can add <code>p.reset()</code> as an event listener to <code>dd</code> which will then clean the entire pipe after a drift was detected. While this solves the problem, it is not very data efficient and results in a long "cool down time." A slightly more efficient solution would be to add the event listener <code>p["reference"].reset()</code> which only cleans the reference window. Due to the dynamic of fixed windows, the data that is currently contained in the sliding window will then become the new reference sample. However, we still need to receive several samples until the reference window is decently filled to ensure high test quality.
<br>
A more efficient strategy is to use part of the current window as reference window, but also keep some to ensure high testing quality even directly after the reset. This can be done using a setup as follows:
<pre>
[reference: fixed 250] <- [current: [slide 125] <- [slide 125]] <-o
</pre>
In case of a reset, we want to move the content of the older part of the current window into the reference window. To accomplish this we first empty the reference window and then push the content of the older current window into the reference window. This translates to a reset script <code>reference,current_2</code>.
<br>
An even more advanced construction that for example uses a growing+static reference window then also requires closing-opening commands:
<pre>
[reference: [fixed 150] <- [growing 100]] <- [current: slide 250] <-o
</pre>
The simple solution <code>reference_1,reference_2,current,reference_1</code> which first empties the growing window, then then fixed window and finally pushes the content of the current window down (in two steps) does not work; since the growing window is smaller than the current window it will emit the samples in an unspecified order so we do not end up with the oldest samples in the fixed window. To solve this the close-open-commands can be used: <code>reference_1(reference_2,current),reference_1</code> first cleans the fixed window, then pushes the samples of the current window right into fixed window and then cleans the growing window.
   <!--p>Topics:
         <ul>
             <li>Discuss reset methods</li>
         </ul></p-->
   </div>
 </div>


*Hint:* Remember that pipes can be named and that the data content of a pipe is the concatenation of the data content of its parts.
</details>
Your code goes here:


In [9]:
my_pipe = Pipeline(Pipeline(GrowingWindow(max_size=150), SlidingWindow(max_size=100), name="ref"),
                   SlidingWindow(max_size=250, name="cur"),
                   WindowChunker(chunk_size=10))

my_detector = MMD(my_pipe["ref"], my_pipe["cur"])


def handle_drift(**payload):
    print(f"Found drift! Score: {payload['score']}, P-Val: {payload['p_value']}")
    my_pipe["ref"].reset()


my_detector.add_listener("drift_detected",handle_drift)

my_stream.add_listener(my_pipe)
my_stream.run()

Found drift! Score: 0.009570961119977489, P-Val: 0.0304
Found drift! Score: 0.03874216142413829, P-Val: 0.0212
Found drift! Score: 0.02366819284847989, P-Val: 0.0092
Found drift! Score: 0.08102379416429376, P-Val: 0.004
Found drift! Score: 0.1424763368011745, P-Val: 0.0028
Found drift! Score: 0.201285117036342, P-Val: 0.032
Found drift! Score: 0.20477287330288385, P-Val: 0.0212
Found drift! Score: 0.30767394431990225, P-Val: 0.0036
Found drift! Score: 0.3492750038140671, P-Val: 0.0024
Found drift! Score: 0.3474110221096308, P-Val: 0.0028
Found drift! Score: 0.40929563060401264, P-Val: 0.0
Found drift! Score: 0.5277697081413162, P-Val: 0.0004
Found drift! Score: 0.5344245238505003, P-Val: 0.0
Found drift! Score: 0.694777316473378, P-Val: 0.0
Found drift! Score: 0.8279851415771573, P-Val: 0.0
Found drift! Score: 0.8228484838265029, P-Val: 0.0
Found drift! Score: 0.8506731238599528, P-Val: 0.0
Found drift! Score: 0.9979903644583078, P-Val: 0.0
Found drift! Score: 1.0147828839042181, P-Val

Self check your code: (Human-based Unit-Test)

In [10]:
# Use this check, when you are ready!
check1_5(my_pipe, my_detector, my_stream)  # Use this check, when you are ready!

This test checks, if your pipeline is correctly implemented and if your MMD implementation found one or some possible drifts.
Great! Your pipeline is correct!
Now checking your stream and MMD...
Found drift! Score: 0.023668192848480176, P-Val: 0.008
Found drift! Score: 0.08102379416429398, P-Val: 0.006
Found drift! Score: 0.14247633680117425, P-Val: 0.004
Found drift! Score: 0.20128511703634244, P-Val: 0.0244
Found drift! Score: 0.20477287330288257, P-Val: 0.0184
Found drift! Score: 0.30767394431990364, P-Val: 0.0036
Found drift! Score: 0.3492750038140683, P-Val: 0.002
Found drift! Score: 0.3474110221096329, P-Val: 0.002
Found drift! Score: 0.4092956306040147, P-Val: 0.0
Found drift! Score: 0.5277697081413152, P-Val: 0.0
Found drift! Score: 0.5344245238504988, P-Val: 0.0
Found drift! Score: 0.6947773164733781, P-Val: 0.0
Found drift! Score: 0.8279851415771602, P-Val: 0.0
Found drift! Score: 0.8228484838265018, P-Val: 0.0
Found drift! Score: 0.8506731238599516, P-Val: 0.0
Found drift! S

## Step 6<sup>*</sup> — Window Annotation Langauge

**You are learning:** How to describe complex data flows and window structur more easily by using DRAGON's Window Annotation Language (WAL) 

**This exercise requires external code [(LARK)](https://github.com/lark-parser/lark) and is completely optional**

You can install LARK by running
```
pip install lark --upgrade
```

Creating complex window setups can get rather cumbersome and leads to spagetti code that is hard to read. Therefore, we came up with a little "programming" language to simplify this. 

**Your task:** Solve step 5 again using WAL. Then add the drift detector as in step 3 and run it again as in step 4.
<details>
<summary>Open Tutorial</summary>
<div class="toggle-box" id="box1">
  <div class="content"  style="padding-left: 3em;">
    <h3>Mini Tutorial on DRAGON's WAL (Window Annotation Language):</h3>
    <p>
    The WAL is a rather simple description language to create standard pipelines for DRAGON. Its syntax is rather simple and graphical.
<ul>
<li>WAL is based on sequences of objects (windows, maps, filters, etc.) that are stringed together</li>
<li><b>Sequences</b> are notated by "arrows" (pointing the the left), i.e., <code>... <- Obj <- ...</code>, here, as in pipes, the data flows from right to left as indicated by the arrow direction</li>
<li><code>Map</code>s are marked by round parentheses <code> (...)</code>. The first keyword in those is interpreted as the functions name --- the according name-to-function directory has to be provided to the parse method ---, following that additional parameters can be provided in a <code>name: parameter</code> notation. The data points can either be placed in a specified argument (<code>at param_name</code>) or by default resolved (similar to the <code>**args</code> notation in python); the result of the Map is either returned as the new data point or added to the data point in a specific argument specified by the <code>put</code> keyword. Example <code>... <- (my_func at x put y type: "abc") <- ... </code> is complied to <code>data["y"] = function_regestry["my_func"](x=data, type="abc")</code> </li>
<li><code>Filter</code>s are marked by angular parentheses <code><...></code>. The syntax is the same as for maps.</li>
<li><code>Chunker</code> are marekd by <code><<size_param)</code> with <code>size_param</code> marking chunk size (integer). Exampel: <code>... <- <<10) <- ...</code> is translated to <code>Chunker(10)</code></li>
<li><code>Fork</code>s are notated as a list of comma separated sequences ended by a pipe and started by a curly parenthesis. Example <code>| seq1 , seq2, ... , seqN } <- ... </code>.</li>
<li>Windows and <code>Pipe</code>s are marked by square parentheses <code>[...]</code>. The window/pipe can be named by adding it at first, i.e., <code>[name: ...]</code>. The remainder depends on the specific type:
<ul>
<li>For pipes the inner part is a sequence with an optional reset code at the end, i.e., <code>[W1 <- W2 <- W3 reset="W2(W1,W3),W2"]</code> or <code>[reference_window: W1 <- W2]</code></li>
<li>For all other types of windows the first inner token is interpreted as the window type followed by additional parameters in a <code>name: parameter</code> notation. Example: <code>[ref: slide max_size: 100]</code> is complied to <code>SlidingWindow(name="ref", max_size=100)</code>. The window type names are listed below
<table border>
<tr><th>short hand<th>long name<th>DRAGON class / Constructor</tr>
<tr><td>slide<td>SlidingWindow<td>...</tr>
<tr><td>grow</td><td>GrowingWindow</td><td></td></tr>
<tr><td>static</td><td>StaticWindow</td><td></td></tr>
</table>
</ul>
</li>
</ul>
  </div>
</div>
</details>
Your code goes here:

In [11]:
my_pipe = parse("[[[grow max_size=150] <- [slide max_size=100]] <- [slide max_size=250] <- <<10) ]")

Cannot parse as LARK (https://github.com/lark-parser/lark) is not installed. Please install LARK via 'pip install lark --upgrade' and try again


In [12]:
# Use this check, when you are ready!
check1_6(my_pipe)

This test checks if your WAL code compiled to the correct pipeline.
Oh no: The parser returned nothing. Check your syntax.
